# Structure of the raw SFTR securities lending data

Source table `crp_sftds_ecb.trade_states_securitieslending`. One row per trade state, reporting side and reference period.

In [ ]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

## Columns

In [ ]:
query = f"""

DESCRIBE crp_sftds_ecb.trade_states_securitieslending

"""
df = pd.read_sql_query(query, cnxn)
df

## Granularity

Reference periods are daily. A trade appears once per day and reporting side, so a trade reported by both sides gives two rows per day.

In [1]:
query = f"""

SELECT reference_period, reporting_cpty_id, other_cpty_id, counterparty_side, uti
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti = '549300GWM9UJLZCGSN04R0MUWSFPU8MPRO8K5P83F2KUVLIFOYYYLY7A86QTJNMN76CPND8WN28Z'
ORDER BY reference_period

"""
df = pd.read_sql_query(query, cnxn)
df['reference_period'].value_counts()

reference_period
2026-06-04    2
2026-07-28    2
2026-07-14    2
2026-07-15    2
2026-07-16    2
2026-07-17    2
2026-07-20    2
2026-07-21    2
2026-07-22    2
2026-07-23    2
2026-07-24    2
2026-07-27    2
2026-07-29    2
2026-07-10    2
2026-07-30    2
2026-07-31    2
2026-08-03    2
2026-08-04    2
2026-08-05    2
2026-08-06    2
2026-08-07    2
2026-08-10    2
2026-08-11    2
2026-08-12    2
2026-07-13    2
2026-07-09    2
2026-06-05    2
2026-06-22    2
2026-06-08    2
2026-06-09    2
2026-06-10    2
2026-06-11    2
2026-06-12    2
2026-06-15    2
2026-06-16    2
2026-06-17    2
2026-06-18    2
2026-06-19    2
2026-06-23    2
2026-07-08    2
2026-06-24    2
2026-06-25    2
2026-06-26    2
2026-06-29    2
2026-06-30    2
2026-07-01    2
2026-07-02    2
2026-07-03    2
2026-07-06    2
2026-07-07    2
2026-08-13    2
Name: count, dtype: int64

## Deduplication key

`tec_ruti` groups the two legs of one trade (it appears to be both counterparty LEIs followed by the UTI). `tec_best_value_leg` is set only within pairs.

In [2]:
query = f"""

SELECT n_rows, n_best, n_dates, COUNT(*) AS n_groups
FROM (
  SELECT tec_ruti, reference_period,
         COUNT(*) AS n_rows,
         SUM(CASE WHEN tec_best_value_leg = 1 THEN 1 ELSE 0 END) AS n_best,
         COUNT(DISTINCT event_date) AS n_dates
  FROM crp_sftds_ecb.trade_states_securitieslending
  GROUP BY tec_ruti, reference_period
) g
GROUP BY n_rows, n_best, n_dates
ORDER BY n_groups DESC

"""
df = pd.read_sql_query(query, cnxn)
df.head(30)

,n_rows,n_best,n_dates,n_groups
0,1,0,1,121625857
1,2,1,1,16065709
2,2,1,2,4972640
3,1,1,1,1868
4,3,0,2,78
5,139230,0,956,1
6,138687,0,965,1
7,139772,0,953,1
8,139474,0,960,1
9,133509,0,941,1


Rows without `tec_ruti` are net exposure collateral updates, one block per day, without UTI or ISIN.

In [3]:
query = f"""

SELECT reference_period, action_type,
       CASE WHEN uti IS NULL THEN 1 ELSE 0 END AS uti_missing,
       CASE WHEN loan_security_id IS NULL THEN 1 ELSE 0 END AS isin_missing,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NULL OR tec_ruti = ''
GROUP BY 1, 2, 3, 4
ORDER BY reference_period, n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

,reference_period,action_type,uti_missing,isin_missing,n
0,2026-06-01,COLU,1,1,129554
1,2026-06-02,COLU,1,1,130317
2,2026-06-03,COLU,1,1,130625
3,2026-06-04,COLU,1,1,132193
4,2026-06-05,COLU,1,1,127244
...,...,...,...,...,...
73,2026-09-10,COLU,1,1,134582
74,2026-09-11,COLU,1,1,133509
75,2026-09-14,COLU,1,1,130676
76,2026-09-15,COLU,1,1,130152


`tec_surrogate_key` is unique per row and serves as join key for the array aggregates.

In [4]:
query = f"""

SELECT COUNT(*), COUNT(DISTINCT tec_surrogate_key)
FROM crp_sftds_ecb.trade_states_securitieslending

"""
df = pd.read_sql_query(query, cnxn)
df

,expr_1,expr_2
0,174315664,174315664


## Collateral arrays

Collateral sits in the array columns `collateral_security` and `collateral_cash`. Net exposure loans carry nothing on the loan row.

In [5]:
query = f"""

SELECT collateralisation_net_exposure, uncollateralised_flag,
       CASE WHEN number_collateral_securities > 0 THEN 1 ELSE 0 END AS has_sec,
       CASE WHEN number_collateral_cash > 0 THEN 1 ELSE 0 END AS has_cash,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NOT NULL AND tec_ruti <> ''
GROUP BY 1, 2, 3, 4
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

,collateralisation_net_exposure,uncollateralised_flag,has_sec,has_cash,n
0,True,False,0,0,115704011
1,False,False,0,1,31033129
2,None,True,0,0,10672004
3,False,False,0,0,2216146
4,False,False,1,0,1813612
5,True,False,1,0,1194875
6,True,False,0,1,1038146
7,False,False,1,1,19174
8,None,False,0,0,9917
9,True,False,1,1,3643


Trade level loans without arrays are mostly basket collateral.

In [6]:
query = f"""

SELECT CASE WHEN collateral_basket_id IS NULL THEN 0 ELSE 1 END AS has_basket, COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NOT NULL AND tec_ruti <> ''
  AND collateralisation_net_exposure = FALSE AND uncollateralised_flag = FALSE
  AND COALESCE(number_collateral_securities, 0) = 0
  AND COALESCE(number_collateral_cash, 0) = 0
GROUP BY 1

"""
df = pd.read_sql_query(query, cnxn)
df

,has_basket,n
0,1,2087106
1,0,129040


## Rebate rate

For floating rates `rebate_rate_derived_sdw` equals index plus spread, in percent per annum.

In [7]:
query = f"""

SELECT rebate_rate_type, COUNT(*) AS n,
       APPX_MEDIAN(rebate_rate_derived_sdw
                   - (flt_rebate_rate_value_sdw + flt_rebate_rate_spread_basispoints / 100)) AS med_diff,
       MIN(rebate_rate_derived_sdw) AS min_rate, MAX(rebate_rate_derived_sdw) AS max_rate
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE flt_rebate_rate IS NOT NULL
GROUP BY 1

"""
df = pd.read_sql_query(query, cnxn)
df

,rebate_rate_type,n,med_diff,min_rate,max_rate
0,Floating,1180844,0.0,3.3,11.56


The cleaning query built from these checks is in `sec_lending_clean_query.txt`.